# Tune de Hiperparametros das GANs — avaliacao 10-fold OOF do LSTM

**Objetivo honesto:** explorar hiperparametros das GANs (GAN, WGAN-GP, cWGAN-GP, CTGAN) sem tocar
na Tabela II do artigo (protocolo fixo). Cada config e avaliada com **StratifiedKFold(10) out-of-fold**
sobre o mesmo LSTM/sample do artigo, e o resultado vai para `tune_oof10_<dataset>.csv`
(media +/- desvio de ACC/Recall/F1 + FN total) + tempo de treino do gerador (TT).

**Protocolo (sem vazamento, espelhado no OOF-10 do artigo):**
- scaler/seletor ajustados somente no treino de cada dobra;
- GAN treinada UMA vez na amostra minoritaria global (inferencia por dobra) — mesmo convenio do artigo;
- LSTM retreinada em cada dobra (parada antecipada);
- media +/- desvio sobre as 10 dobras + FN total.

**Como usar:** rode as celulas acima (download + pre-processamento) ate gerar `X_s,y_s,NB_DATASET`,
depois ajuste `TUNE_GRID` na penultima celula (para uma passada rapida, deixe 1 config por familia)
e execute as tres celulas finais.

**Deixe a Tabela II do artigo intacta: este notebook e um estudo de sensibilidade/custo.**


In [ ]:
# ========================================
# CELULA 1: Instala dependencias + seeds
# ========================================
!pip install -q imbalanced-learn xgboost tensorflow scikit-learn matplotlib seaborn ctgan pyarrow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

np.random.seed(42)
tf.random.set_seed(42)
print("Dependencias carregadas!")


In [ ]:
# ========================================
# CELULA 2: Helpers de download (com fallback)
# ========================================
import os, time
from urllib.request import urlopen, Request

def baixar(url, destino, tentativas=3):
    """Baixa com header de navegador e progresso simples; tenta N vezes."""
    req = Request(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "*/*"})
    for t in range(1, tentativas + 1):
        try:
            with urlopen(req, timeout=120) as r:
                total = int(r.headers.get("Content-Length") or 0)
                lido = 0
                with open(destino, "wb") as f:
                    while True:
                        bloco = r.read(1 << 20)
                        if not bloco:
                            break
                        f.write(bloco)
                        lido += len(bloco)
                        if total:
                            pct = 100 * lido / total
                            print(f"\r  {lido/1e6:.1f}/{total/1e6:.0f} MB ({pct:.0f}%)", end="")
            print("")
            print(f"  OK: {destino} ({os.path.getsize(destino)/1e6:.1f} MB)")
            return True
        except Exception as e:
            print(f"  Tentativa {t} falhou: {e}")
            time.sleep(3)
    return False


In [ ]:
# ========================================
# CELULA 3: Download dos 3 parquets reais do IoT-23 (HuggingFace resolve)
# ========================================
BASE = "https://huggingface.co/datasets/19kmunz/iot-23-preprocessed-allcolumns/resolve/main/data/"
ARQUIVOS = [
    ("train-00000-of-00003-47134d3a10206e3c.parquet", "train-00000.parquet"),
    ("train-00001-of-00003-7fb0937ba34dd762.parquet", "train-00001.parquet"),
    ("train-00002-of-00003-78be3fca5c692525.parquet", "train-00002.parquet"),
]

os.makedirs("/content/iot23_real", exist_ok=True)
for origem, destino_nome in ARQUIVOS:
    destino = "/content/iot23_real/" + destino_nome
    if os.path.exists(destino) and os.path.getsize(destino) > 1e7:
        print("Ja presente:", destino)
        continue
    print("Baixando:", origem)
    if not baixar(BASE + origem, destino):
        print("Falhou:", origem, "-> baixe manualmente os 3 .parquet e coloque em /content/iot23_real/")

import glob
pres = sorted(glob.glob("/content/iot23_real/*.parquet"))
if len(pres) < 3:
    print("\nATENCAO: faltam parquets. Baixe os 3 arquivos do HF manualmente.")
    raise SystemExit("Download incompleto.")

print("\nArquivos presentes:")
for p in pres:
    print("  ", os.path.basename(p), f"{os.path.getsize(p)/1e6:.0f} MB")

# Leitura + concat com baixo uso de RAM
frame_list = []
for p in pres:
    f = pd.read_parquet(p)
    frame_list.append(f)
    print(f"  {os.path.basename(p)}: {len(f)} linhas, {f.shape[1]} colunas")

import gc
df = pd.concat(frame_list, ignore_index=True, sort=False)
del frame_list
gc.collect()
print("\nIoT-23 real total:", df.shape)


In [ ]:
# ========================================
# Funcao de limpeza do IoT-23 real (Zeek conn.log)
# ========================================
def limpar_iot23(df):
    """Deixa apenas colunas numericas de fluxo + label binario (0=Benigno, 1=Malicioso)."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    antes = len(df)
    df = df[df['label'].notna()]
    lab = df['label'].astype(str).str.strip().str.lower()
    df['y'] = (~lab.isin(['benign'])).astype(int)
    # descarta identidade / timestamp / colunas de texto do Zeek
    drop = ['ts', 'uid', 'id.orig_h', 'id.resp_h', 'proto', 'service',
            'conn_state', 'local_orig', 'local_resp', 'history']
    drop = [c for c in drop if c in df.columns]
    df = df.drop(columns=drop)
    X = df.drop(columns=['label', 'y']).apply(pd.to_numeric, errors='coerce')
    X = X.dropna(axis=1, how='all')
    X = X.loc[:, X.notna().mean() > 0.7]
    X = X.loc[:, X.nunique() > 1]
    X = X.fillna(0).replace([np.inf, -np.inf], 0)
    X = X.astype('float32')
    y = df['y'].values.astype(int)
    print("  Linhas: %d -> %d após limpeza." % (antes, len(df)))
    return X, y
# ========================================
# Funcoes compartilhadas de limpeza / amostragem
# ========================================
def limpar_tabular(df, col_label, mapear):
    """Deixa apenas colunas numericas + label binario."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    col = col_label.lower()
    antes = len(df)
    if df[col].dtype == object:
        # normaliza hifens unicode (en-dash/em-dash) usados no CICIDS2017
        df[col] = df[col].str.replace("\u2013", "-", regex=True).str.replace("\u2014", "-", regex=True)
        # remove linhas de cabecalho repetido (label igual ao nome da coluna)
        df = df[df[col].astype(str).str.strip().str.lower() != col]
    # descarta linhas sem label e valores de lixo que viram 'nan'
    df = df[df[col].notna()]
    df[col] = df[col].astype(str).str.strip().str.lower()
    df = df[~df[col].isin(["nan", "none", "na", "n/a", "-", "label"])]
    # mapeia e REMOVE (com aviso) qualquer label nao mapeado - nao aborta mais
    df['label'] = df[col].map(mapear)
    nao_mapeadas = df[df['label'].isna()]
    if len(nao_mapeadas):
        print("  AVISO limpar_tabular: %d linhas removidas por label nao mapeado/NaN." % len(nao_mapeadas))
        unicos = nao_mapeadas[col].unique()[:10]
        print("  Labels removidos:", unicos)
        df = df[df['label'].notna()]
    print("  Linhas: %d -> %d após limpeza (%d removidas)." % (antes, len(df), antes - len(df)))
    if col != 'label':
        df = df.drop(columns=[col])
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    X = df.drop(columns=['label']).apply(pd.to_numeric, errors='coerce')
    X = X.dropna(axis=1, how='all')
    X = X.loc[:, X.notna().mean() > 0.7]
    X = X.loc[:, X.nunique() > 1]
    X = X.fillna(0)
    X = X.replace([np.inf, -np.inf], 0)
    # compacta para float32: corta a RAM pela metade. Valores que estouram a
    # faixa do float32 viram inf no cast -> zerados na sequencia (mesma
    # politica de tratamento de inf adotada acima).
    import warnings as _w
    with _w.catch_warnings():
        _w.simplefilter('ignore', RuntimeWarning)
        X = X.astype('float32')
    X = X.replace([np.inf, -np.inf], 0)
    y = df['label'].astype(int).values
    del df
    return X, y

def nb_amostra_estratificada(X, y, n_total, frac_benigno, seed=42):
    """Amostra estratificada para espelhar o protocolo do IoT-23 (82% benigno / 18% ataque)."""
    n_ben = int(n_total * frac_benigno)
    n_att = n_total - n_ben
    n_ben_disp = int(sum(y == 0)); n_att_disp = int(sum(y == 1))
    if n_ben > n_ben_disp or n_att > n_att_disp:
        raise ValueError(
            f"Populacao insuficiente para 82/18: disponivel benigno={n_ben_disp}, "
            f"ataque={n_att_disp}; necessario benigno={n_ben}, ataque={n_att}. "
            f"Verifique se TODOS os arquivos do dataset foram baixados.")
    idx_ben = np.random.RandomState(seed).choice(np.where(y == 0)[0], n_ben, replace=False)
    idx_att = np.random.RandomState(seed).choice(np.where(y == 1)[0], n_att, replace=False)
    idx = np.concatenate([idx_ben, idx_att])
    return X.iloc[idx].reset_index(drop=True), y[idx]


In [ ]:
# ========================================
# CELULA 4: Pre-processamento + amostragem 40k (82/18) + SelectPercentile
# ========================================
N_TOTAL = 40000
FRAC_BENIGNO = 0.82
PERCENTIL = 60
NB_DATASET = "iot23_real"

X, y = limpar_iot23(df)

print(f"Registros unicos limpos: {len(X)} | Features numericas: {X.shape[1]}")
print(f"Distribuicao real: benigno={int(sum(y==0))} ({sum(y==0)/len(y):.1%}) | ataque={int(sum(y==1))} ({sum(y==1)/len(y):.1%})")

X_s, y_s = nb_amostra_estratificada(X, y, N_TOTAL, FRAC_BENIGNO)
print(f"\nAmostra final: {len(X_s)} | benigno={int(sum(y_s==0))} ({sum(y_s==0)/len(y_s):.1%}) | ataque={int(sum(y_s==1))} ({sum(y_s==1)/len(y_s):.1%})")
print('AmosTrA pronta para o tune:', X_s.shape, 'ataque=', int(y_s.sum()))


In [ ]:
# ================================================
# TUNE - builders GAN parametrizados (mesma arq. da aula)
# ================================================
import time as _t
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from imblearn.combine import SMOTETomek
from tensorflow.keras.callbacks import EarlyStopping

def criar_lstm(n_feat):
    model = models.Sequential(name="lstm_classifier")
    model.add(layers.Input(shape=(n_feat, 1)))
    model.add(layers.LSTM(64, return_sequences=True))
    model.add(layers.Dropout(0.3))
    model.add(layers.LSTM(32))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

def treina_gan(X_min, n_dim, noise_dim=32, epochs=300, batch=64):
    def criar_gerador():
        m = models.Sequential()
        m.add(layers.Dense(64, input_dim=noise_dim, activation="relu")); m.add(layers.BatchNormalization())
        m.add(layers.Dense(128, activation="relu")); m.add(layers.BatchNormalization())
        m.add(layers.Dense(256, activation="relu")); m.add(layers.Dense(n_dim, activation="tanh"))
        return m
    def criar_disc():
        m = models.Sequential()
        m.add(layers.Dense(256, input_dim=n_dim, activation="relu")); m.add(layers.Dropout(0.3))
        m.add(layers.Dense(128, activation="relu")); m.add(layers.Dropout(0.3))
        m.add(layers.Dense(1, activation="sigmoid"))
        return m
    ger = criar_gerador(); dis = criar_disc()
    dis.compile(optimizer=tf.keras.optimizers.Adam(0.0002), loss="binary_crossentropy")
    dis.trainable = False
    gan = models.Sequential([ger, dis])
    gan.compile(optimizer=tf.keras.optimizers.Adam(0.0002), loss="binary_crossentropy")
    X_t = tf.convert_to_tensor(X_min, dtype=tf.float32)
    meio = batch // 2
    ds = tf.data.Dataset.from_tensor_slices(X_t).shuffle(1000).batch(meio)
    t0 = _t.time()
    for ep in range(epochs):
        for b in ds:
            reais = tf.ones((b.shape[0], 1)); ru = tf.random.normal((b.shape[0], noise_dim))
            fals = ger(ru, training=True); labf = tf.zeros((b.shape[0], 1))
            dis.trainable = True; dis.train_on_batch(b, reais); dis.train_on_batch(fals, labf)
            dis.trainable = False
            ru2 = tf.random.normal((batch, noise_dim)); eng = tf.ones((batch, 1))
            gan.train_on_batch(ru2, eng)
        if (ep + 1) % 100 == 0:
            print(f"    GAN ep {ep+1}/{epochs}")
    return ger, _t.time() - t0

def treina_wgan(X_min, n_dim, noise_dim=32, epochs=300, batch=64, n_critic=5, lmbda=10.0):
    def criar_gerador_wgan():
        m = models.Sequential()
        m.add(layers.Dense(64, input_dim=noise_dim, activation="relu")); m.add(layers.BatchNormalization())
        m.add(layers.Dense(128, activation="relu")); m.add(layers.BatchNormalization())
        m.add(layers.Dense(256, activation="relu")); m.add(layers.Dense(n_dim, activation="tanh"))
        return m
    def criar_critico():
        m = models.Sequential()
        m.add(layers.Dense(256, input_dim=n_dim, activation="relu")); m.add(layers.Dropout(0.3))
        m.add(layers.Dense(128, activation="relu")); m.add(layers.Dropout(0.3))
        m.add(layers.Dense(1, activation="linear"))
        return m
    critico = criar_critico(); ger = criar_gerador_wgan()
    c_opt = tf.keras.optimizers.Adam(0.0002, beta_1=0.5, beta_2=0.9)
    g_opt = tf.keras.optimizers.Adam(0.0002, beta_1=0.5, beta_2=0.9)
    X_t = tf.convert_to_tensor(X_min, dtype=tf.float32); n_min = X_min.shape[0]
    def gp(real, fake):
        alpha = tf.random.uniform((tf.shape(real)[0], 1), 0.0, 1.0)
        interp = alpha * real + (1.0 - alpha) * fake
        with tf.GradientTape() as tape:
            tape.watch(interp); pred = critico(interp, training=True)
        grads = tape.gradient(pred, interp)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
        return tf.reduce_mean((norm - 1.0) ** 2)
    def batch_real(tam):
        idx = np.random.randint(0, n_min, tam); return tf.gather(X_t, idx)
    t0 = _t.time()
    for ep in range(epochs):
        for _ in range(n_critic):
            reais = batch_real(batch); ru = tf.random.normal((batch, noise_dim))
            with tf.GradientTape() as tape:
                f = ger(ru, training=True)
                w = tf.reduce_mean(critico(f, training=True)) - tf.reduce_mean(critico(reais, training=True)) + lmbda * gp(reais, f)
            grd = tape.gradient(w, critico.trainable_variables)
            c_opt.apply_gradients(zip(grd, critico.trainable_variables))
        ru = tf.random.normal((batch, noise_dim))
        with tf.GradientTape() as tape:
            f = ger(ru, training=True); gl = -tf.reduce_mean(critico(f, training=True))
        grd = tape.gradient(gl, ger.trainable_variables)
        g_opt.apply_gradients(zip(grd, ger.trainable_variables))
        if (ep + 1) % 100 == 0:
            print(f"    WGAN ep {ep+1}/{epochs}")
    return ger, _t.time() - t0

def treina_cwgan(X_all, y_all, n_dim, noise_dim=32, epochs=300, batch=64, n_critic=5, lmbda=10.0):
    def criar_gerador_cwgan():
        ru = layers.Input(shape=(noise_dim,)); rot = layers.Input(shape=(1,))
        rc = layers.Lambda(lambda x: tf.cast(x, tf.int32))(rot)
        emb = layers.Flatten()(layers.Embedding(2, 16)(rc))
        x = layers.Concatenate()([ru, emb])
        x = layers.Dense(64, activation="relu")(x); x = layers.BatchNormalization()(x)
        x = layers.Dense(128, activation="relu")(x); x = layers.BatchNormalization()(x)
        x = layers.Dense(256, activation="relu")(x)
        return models.Model([ru, rot], layers.Dense(n_dim, activation="tanh")(x), name="gcw")
    def criar_critico_cwgan():
        am = layers.Input(shape=(n_dim,)); rot = layers.Input(shape=(1,))
        rc = layers.Lambda(lambda x: tf.cast(x, tf.int32))(rot)
        emb = layers.Flatten()(layers.Embedding(2, 16)(rc))
        x = layers.Concatenate()([am, emb])
        x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(0.3)(x)
        x = layers.Dense(128, activation="relu")(x); x = layers.Dropout(0.3)(x)
        return models.Model([am, rot], layers.Dense(1, activation="linear")(x), name="ccw")
    ger = criar_gerador_cwgan(); crit = criar_critico_cwgan()
    c_opt = tf.keras.optimizers.Adam(0.0002, beta_1=0.5, beta_2=0.9)
    g_opt = tf.keras.optimizers.Adam(0.0002, beta_1=0.5, beta_2=0.9)
    X_t = tf.convert_to_tensor(X_all, dtype=tf.float32)
    y_t = tf.convert_to_tensor(y_all.reshape(-1, 1), dtype=tf.float32)
    n_all = X_all.shape[0]
    def gp(real, fake, rot):
        alpha = tf.random.uniform((tf.shape(real)[0], 1), 0.0, 1.0)
        interp = alpha * real + (1.0 - alpha) * fake
        with tf.GradientTape() as tape:
            tape.watch(interp); pred = crit([interp, rot], training=True)
        grads = tape.gradient(pred, interp)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
        return tf.reduce_mean((norm - 1.0) ** 2)
    def batch_real(tam):
        idx = np.random.randint(0, n_all, tam)
        return tf.gather(X_t, idx), tf.gather(y_t, idx)
    t0 = _t.time()
    for ep in range(epochs):
        for _ in range(n_critic):
            reais, rot = batch_real(batch); ru = tf.random.normal((batch, noise_dim))
            with tf.GradientTape() as tape:
                f = ger([ru, rot], training=True)
                w = tf.reduce_mean(crit([f, rot], training=True)) - tf.reduce_mean(crit([reais, rot], training=True)) + lmbda * gp(reais, f, rot)
            grd = tape.gradient(w, crit.trainable_variables)
            c_opt.apply_gradients(zip(grd, crit.trainable_variables))
        ru = tf.random.normal((batch, noise_dim)); rot_g = tf.random.uniform((batch, 1), 0, 2, dtype=tf.float32)
        with tf.GradientTape() as tape:
            f = ger([ru, rot_g], training=True); gl = -tf.reduce_mean(crit([f, rot_g], training=True))
        grd = tape.gradient(gl, ger.trainable_variables)
        g_opt.apply_gradients(zip(grd, ger.trainable_variables))
        if (ep + 1) % 100 == 0:
            print(f"    cWGAN ep {ep+1}/{epochs}")
    return ger, _t.time() - t0

def treina_ctgan(X_min, epochs=300, batch_size=200, pac=10):
    df_min = pd.DataFrame(X_min, columns=[f"f{i}" for i in range(X_min.shape[1])])
    from ctgan import CTGAN
    m = CTGAN(epochs=epochs, batch_size=batch_size, pac=pac, verbose=False)
    t0 = _t.time(); m.fit(df_min)
    return m, _t.time() - t0


In [ ]:
# ================================================
# TUNE - avaliacao StratifiedKFold(10) OOF (LSTM)
# ================================================
def oof10_lstm(X_s, y_s, modo, noise_dim=32, lstm_epochs=60, seed=42):
    """modo = 'original' | 'smt' | funcao(Xmin_fold, n_sint, noise_dim)->(n_sint, nfeat)"""
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    X_o = np.asarray(X_s, dtype=np.float32); y_o = np.asarray(y_s)
    linhas = []
    for fold, (tr, va) in enumerate(skf.split(X_o, y_o), 1):
        X_tr, Y_tr = X_o[tr], y_o[tr]; X_va, Y_va = X_o[va], y_o[va]
        sc = StandardScaler().fit(X_tr); X_ts = sc.transform(X_tr); X_vs = sc.transform(X_va)
        sel = SelectPercentile(f_classif, percentile=60).fit(X_ts, Y_tr)
        X_ts = sel.transform(X_ts); X_vs = sel.transform(X_vs)
        nf = X_ts.shape[1]
        nat = int(Y_tr.sum()); nben = int((1 - Y_tr).sum())
        X_b, Y_b = X_ts, Y_tr
        n_sint = nben - nat
        if modo == "original":
            pass
        elif modo == "smt":
            X_b, Y_b = SMOTETomek(random_state=42).fit_resample(X_ts, Y_tr)
        elif n_sint > 0:
            sint = np.asarray(modo(X_ts[Y_tr == 1], n_sint, noise_dim), dtype=np.float32)
            if sint.ndim == 3:
                sint = sint.reshape(len(sint), -1)
            X_b = np.vstack([X_ts, sint[:, :nf]])
            Y_b = np.hstack([Y_tr, np.ones(n_sint)])
        lstm = criar_lstm(nf)
        es = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
        lstm.fit(X_b.reshape(-1, nf, 1), Y_b, validation_split=0.15, epochs=lstm_epochs,
                 batch_size=64, callbacks=[es], verbose=0)
        yp = (lstm.predict(X_vs.reshape(-1, nf, 1), verbose=0) > 0.5).astype(int).ravel()
        tn, fp, fn, tp = confusion_matrix(Y_va, yp).ravel()
        linhas.append({"Fold": fold, "ACC": accuracy_score(Y_va, yp),
                       "Recall": recall_score(Y_va, yp), "F1": f1_score(Y_va, yp),
                       "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)})
    d = pd.DataFrame(linhas)
    return dict(ACC_m=round(d.ACC.mean(), 4), ACC_dp=round(d.ACC.std(), 4),
                Rec_m=round(d.Recall.mean(), 4), Rec_dp=round(d.Recall.std(), 4),
                F1_m=round(d.F1.mean(), 4), F1_dp=round(d.F1.std(), 4),
                FN=int(d.FN.sum()), FP=int(d.FP.sum())), d


In [ ]:
# ================================================
# TUNE - grid e loop (10-fold OOF). EDITAR p/ reduzir/aumentar.
# Para uma passada rapida: deixe 1 config por familia.
# ================================================
def _mk_gerador(ger, nd):
    def _f(Xmn, n, nd_=nd, _g=ger):
        r = tf.random.normal((n, nd_))
        return _g(r, training=False).numpy()
    return _f

X_min_full = np.asarray(X_s[y_s == 1], dtype=np.float32)
y_all = y_s
noise_dim = 32

TUNE_GRID = {
    "Original":  [("base", {"noise_dim": noise_dim})],
    "SMOTETomek":[("smt", {})],
    "GAN": [
        ("e300_b64",   {"epochs": 300, "batch": 64}),
        ("e450_b128",  {"epochs": 450, "batch": 128}),
    ],
    "WGAN-GP": [
        ("e300_b64_nc5_l10",  {"epochs": 300, "batch": 64, "n_critic": 5, "lmbda": 10.0}),
        ("e450_b128_nc5_l10", {"epochs": 450, "batch": 128, "n_critic": 5, "lmbda": 10.0}),
        ("e300_b128_nc3_l5",  {"epochs": 300, "batch": 128, "n_critic": 3, "lmbda": 5.0}),
    ],
    "cWGAN-GP": [
        ("e300_b64_nc5_l10",  {"epochs": 300, "batch": 64, "n_critic": 5, "lmbda": 10.0}),
        ("e450_b128_nc5_l10", {"epochs": 450, "batch": 128, "n_critic": 5, "lmbda": 10.0}),
        ("e300_b128_nc3_l5",  {"epochs": 300, "batch": 128, "n_critic": 3, "lmbda": 5.0}),
    ],
    "CTGAN": [
        ("e300_b200", {"epochs": 300, "batch_size": 200}),
        ("e500_b200", {"epochs": 500, "batch_size": 200}),
    ],
}

linhas_tune = []
detalhe = {}
for fam, cfgs in TUNE_GRID.items():
    for nome, cfg in cfgs:
        t0 = _t.time()
        if fam == "Original":
            modo = "original"; nd = cfg["noise_dim"]
        elif fam == "SMOTETomek":
            modo = "smt"; nd = noise_dim
        elif fam == "GAN":
            ger, _ = treina_gan(X_min_full, X_s.shape[1], noise_dim=noise_dim, **cfg)
            modo = _mk_gerador(ger, noise_dim); nd = noise_dim
        elif fam == "WGAN-GP":
            ger, _ = treina_wgan(X_min_full, X_s.shape[1], noise_dim=noise_dim, **cfg)
            modo = _mk_gerador(ger, noise_dim); nd = noise_dim
        elif fam == "cWGAN-GP":
            ger, _ = treina_cwgan(np.asarray(X_s, dtype=np.float32), y_all,
                                  X_s.shape[1], noise_dim=noise_dim, **cfg)
            def _cw(Xmn, n, nd_, _g=ger):
                r = tf.random.normal((n, nd_)); rot = tf.ones((n, 1), dtype=tf.float32)
                return _g([r, rot], training=False).numpy()
            modo = _cw; nd = noise_dim
        elif fam == "CTGAN":
            mdl, _ = treina_ctgan(X_min_full, **cfg)
            def _ct(Xmn, n, nd_, _m=mdl):
                return _m.sample(n).values
            modo = _ct; nd = noise_dim
        TT = _t.time() - t0
        agg, _ = oof10_lstm(X_s, y_s, modo, noise_dim=nd, lstm_epochs=60)
        linhas_tune.append({"Dataset": NB_DATASET, "Familia": fam, "Config": nome,
                            "TT_s": round(TT, 1), **agg})
        print(f"{fam:10s} {nome:22s} TT={TT:7.0f}s FN={agg['FN']:5d} F1={agg['F1_m']:.4f}±{agg['F1_dp']:.4f}")

df_tune = pd.DataFrame(linhas_tune)
CSV_TUNE = f"tune_oof10_{NB_DATASET}.csv"
df_tune.to_csv(CSV_TUNE, index=False)
print("\n=== TUNE-OOF10 =========")
print(df_tune.to_string(index=False))
print("\n=== Melhor F1 por familia ===")
idx = df_tune.groupby("Familia")["F1_m"].idxmax()
print(df_tune.loc[idx][["Familia", "Config", "F1_m", "FN", "TT_s"]].to_string(index=False))
try:
    from google.colab import files
    files.download(CSV_TUNE)
except Exception:
    print("\nBaixe manualmente:", CSV_TUNE)
